In [ ]:
# Installation de kagglehub (téléchargement du dataset depuis Kaggle)
!pip install kagglehub

# TP3 — Régression des émissions de CO2 (CO2 Emissions Canada)

**Objectif :** prédire les émissions de CO2 des véhicules à partir de leurs caractéristiques techniques.

**Approche différente :** on **retire la variable CO2 des features d'entrée** pour éviter toute circularité (le CO2 est la cible, pas un prédicteur). On compare ensuite **trois modèles** de natures différentes :
- un MLP (PyTorch),
- une forêt aléatoire (scikit-learn),
- un LSTM léger (PyTorch), sur une séquence artificielle construite à partir des features.

Enfin, on exporte le MLP au format **TFLite** (TinyML) via un modèle Keras équivalent.

> Dataset attendu : `CO2 Emissions_Canada.csv` (Kaggle). Adapter le chemin si besoin.

## 1. Imports et configuration

On fixe la graine (`seed=42`) pour la reproductibilité et on importe toutes les bibliothèques.

In [ ]:
# Bibliothèques de base
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# scikit-learn : preprocessing, modèle RF et métriques
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# PyTorch : MLP et LSTM
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

# On fixe toutes les graines pour la reproductibilité
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"PyTorch {torch.__version__} | device : {device}")

## 2. Chargement et analyse exploratoire (EDA)

On charge le CSV puis on explore : distribution du CO2, corrélations entre variables numériques, et boxplots des émissions par type de carburant.

In [ ]:
# Téléchargement du dataset CO2 Emissions Canada via kagglehub
import kagglehub
import os

path = kagglehub.dataset_download("debajyotipodder/co2-emission-by-vehicles")
csv_file = [f for f in os.listdir(path) if f.endswith('.csv')][0]
df = pd.read_csv(os.path.join(path, csv_file))
print(f"Dataset chargé : {df.shape}")
print(df.columns.tolist())
df.head()

In [ ]:
# Informations sur les types de colonnes et les valeurs manquantes
df.info()

In [ ]:
# Renommage des colonnes vers des noms courts et homogènes
# (les noms exacts du CSV Kaggle contiennent des unités entre parenthèses)
rename_map = {
    "Engine Size(L)": "Engine Size",
    "Fuel Consumption City (L/100 km)": "Fuel Consumption City",
    "Fuel Consumption Hwy (L/100 km)": "Fuel Consumption Hwy",
    "CO2 Emissions(g/km)": "CO2 Emissions",
}
df = df.rename(columns=rename_map)

# Vérification des valeurs manquantes et du type des colonnes
print(df.isna().sum())
df.describe()

In [ ]:
# Distribution de la cible : émissions de CO2
plt.figure(figsize=(8, 4))
sns.histplot(df["CO2 Emissions"], kde=True, bins=40)
plt.title("Distribution des émissions de CO2 (g/km)")
plt.xlabel("CO2 Emissions (g/km)")
plt.tight_layout()
plt.show()

In [ ]:
# Matrice de corrélation entre les variables numériques
num_cols = ["Engine Size", "Cylinders", "Fuel Consumption City",
            "Fuel Consumption Hwy", "CO2 Emissions"]
plt.figure(figsize=(8, 6))
sns.heatmap(df[num_cols].corr(), annot=True, fmt=".2f", cmap="coolwarm", center=0)
plt.title("Corrélations entre variables numériques")
plt.tight_layout()
plt.show()

In [ ]:
# Boxplots des émissions de CO2 par type de carburant
plt.figure(figsize=(9, 5))
sns.boxplot(data=df, x="Fuel Type", y="CO2 Emissions")
plt.title("Émissions de CO2 par type de carburant")
plt.tight_layout()
plt.show()

## 3. Préprocessing

- **Features retenues** : `Engine Size`, `Cylinders`, `Fuel Consumption City`, `Fuel Consumption Hwy` (numériques) + `Fuel Type` et `Transmission` (one-hot).
- **Cible** : `CO2 Emissions` uniquement (on ne l'utilise jamais comme feature → pas de circularité).
- **StandardScaler** appliqué sur les features numériques uniquement.

In [ ]:
# Définition des colonnes numériques et catégorielles
numeric_features = ["Engine Size", "Cylinders", "Fuel Consumption City", "Fuel Consumption Hwy"]
categorical_features = ["Fuel Type", "Transmission"]

# Encodage one-hot des variables catégorielles
df_encoded = pd.get_dummies(df[categorical_features], prefix=categorical_features)

# Construction de la matrice de features X (numériques + one-hot) et de la cible y
X = pd.concat([df[numeric_features], df_encoded], axis=1).astype(np.float32)
y = df["CO2 Emissions"].values.astype(np.float32)

print(f"Nombre de features après one-hot : {X.shape[1]}")
X.head()

In [ ]:
# Découpage train/test (80% / 20%)
X_train_df, X_test_df, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=SEED)

# StandardScaler appliqué UNIQUEMENT aux features numériques (fit sur le train)
scaler = StandardScaler()
X_train = X_train_df.copy()
X_test = X_test_df.copy()
X_train[numeric_features] = scaler.fit_transform(X_train_df[numeric_features])
X_test[numeric_features] = scaler.transform(X_test_df[numeric_features])

# Conversion en tableaux NumPy pour les modèles
X_train = X_train.values.astype(np.float32)
X_test = X_test.values.astype(np.float32)
n_features = X_train.shape[1]
print(f"Train : {X_train.shape} | Test : {X_test.shape}")

## 4. Modèle 1 — MLP (PyTorch)

Perceptron multicouche à 3 couches cachées (128 → 64 → 32), avec `ReLU`, `BatchNorm` et `Dropout(0.2)` à chaque couche.

In [ ]:
class MLPRegressor(nn.Module):
    """MLP de régression : 3 couches cachées (128->64->32) avec BatchNorm, ReLU et Dropout."""
    def __init__(self, in_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, 128), nn.BatchNorm1d(128), nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(128, 64), nn.BatchNorm1d(64), nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(64, 32), nn.BatchNorm1d(32), nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(32, 1),  # sortie scalaire (régression)
        )

    def forward(self, x):
        return self.net(x).squeeze(1)  # (batch,1) -> (batch,)


# Préparation des DataLoaders PyTorch
train_ds = TensorDataset(torch.tensor(X_train), torch.tensor(y_train))
test_ds = TensorDataset(torch.tensor(X_test), torch.tensor(y_test))
train_dl = DataLoader(train_ds, batch_size=64, shuffle=True)
test_dl = DataLoader(test_ds, batch_size=64, shuffle=False)

In [ ]:
# Instanciation du modèle, de la loss (MSE) et de l'optimiseur Adam
mlp = MLPRegressor(n_features).to(device)
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(mlp.parameters(), lr=1e-3, weight_decay=1e-5)

EPOCHS = 80
mlp_losses = []  # historique de la loss d'entraînement

for epoch in range(1, EPOCHS + 1):
    mlp.train()
    running = 0.0
    for xb, yb in train_dl:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        loss = criterion(mlp(xb), yb)  # MSE entre prédiction et cible
        loss.backward()
        optimizer.step()
        running += loss.item() * xb.size(0)
    epoch_loss = running / len(train_ds)
    mlp_losses.append(epoch_loss)
    if epoch % 10 == 0:
        print(f"Epoch {epoch:02d}/{EPOCHS} | MSE train = {epoch_loss:.2f}")

In [ ]:
# Prédictions du MLP sur le test set
mlp.eval()
with torch.no_grad():
    pred_mlp = mlp(torch.tensor(X_test).to(device)).cpu().numpy()

## 5. Modèle 2 — Random Forest (scikit-learn)

Forêt aléatoire de 200 arbres avec `max_depth=10`. Les forêts ne nécessitent pas de mise à l'échelle, mais on réutilise les mêmes données pour une comparaison équitable.

In [ ]:
# Entraînement de la forêt aléatoire (200 arbres, profondeur max 10)
rf = RandomForestRegressor(n_estimators=200, max_depth=10, random_state=SEED, n_jobs=-1)
rf.fit(X_train, y_train)

# Prédictions sur le test set
pred_rf = rf.predict(X_test)

## 6. Modèle 3 — LSTM léger (PyTorch)

Un LSTM attend une séquence. Comme nos données sont tabulaires, on construit une **séquence artificielle** : chaque feature devient un pas de temps (séquence de longueur `n_features`, chaque pas de dimension 1). Le LSTM (1 couche, `hidden=32`) lit cette séquence et prédit le CO2 via une couche linéaire finale.

In [ ]:
class LSTMRegressor(nn.Module):
    """LSTM léger : lit les features comme une séquence (longueur=n_features, dim=1)."""
    def __init__(self, hidden=32):
        super().__init__()
        self.lstm = nn.LSTM(input_size=1, hidden_size=hidden, num_layers=1, batch_first=True)
        self.fc = nn.Linear(hidden, 1)  # régression à partir du dernier état caché

    def forward(self, x):
        # x : (batch, n_features) -> (batch, n_features, 1) : chaque feature = un pas de temps
        x = x.unsqueeze(-1)
        out, (h_n, _) = self.lstm(x)
        # h_n[-1] = dernier état caché de la dernière couche
        return self.fc(h_n[-1]).squeeze(1)


lstm = LSTMRegressor(hidden=32).to(device)
criterion_lstm = nn.MSELoss()
optimizer_lstm = torch.optim.Adam(lstm.parameters(), lr=1e-3)

In [ ]:
# Entraînement du LSTM (mêmes DataLoaders que le MLP)
for epoch in range(1, EPOCHS + 1):
    lstm.train()
    running = 0.0
    for xb, yb in train_dl:
        xb, yb = xb.to(device), yb.to(device)
        optimizer_lstm.zero_grad()
        loss = criterion_lstm(lstm(xb), yb)
        loss.backward()
        optimizer_lstm.step()
        running += loss.item() * xb.size(0)
    if epoch % 10 == 0:
        print(f"Epoch {epoch:02d}/{EPOCHS} | MSE train = {running / len(train_ds):.2f}")

# Prédictions du LSTM sur le test set
lstm.eval()
with torch.no_grad():
    pred_lstm = lstm(torch.tensor(X_test).to(device)).cpu().numpy()

## 7. Comparaison des 3 modèles

On évalue chaque modèle avec **MAE**, **RMSE** et **R²**, puis on regroupe les résultats dans un tableau récapitulatif.

In [ ]:
def evaluate(name, y_true, y_pred):
    """Calcule MAE, RMSE et R2 pour un modèle donné."""
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2 = r2_score(y_true, y_pred)
    return {"Modèle": name, "MAE": mae, "RMSE": rmse, "R²": r2}

# Tableau récapitulatif des trois modèles
results = pd.DataFrame([
    evaluate("MLP", y_test, pred_mlp),
    evaluate("Random Forest", y_test, pred_rf),
    evaluate("LSTM", y_test, pred_lstm),
]).round(3)

results

## 8. Importance des features (Random Forest)

La forêt aléatoire fournit une mesure d'importance des features, utile pour interpréter quelles caractéristiques pilotent les émissions de CO2.

In [ ]:
# Récupération et tri des importances de features
importances = pd.Series(rf.feature_importances_, index=X.columns).sort_values(ascending=True)

plt.figure(figsize=(8, 6))
importances.plot(kind="barh")
plt.title("Importance des features (Random Forest)")
plt.xlabel("Importance")
plt.tight_layout()
plt.show()

## 9. Export TFLite du MLP (TinyML)

On reconstruit un MLP **équivalent en Keras** (mêmes dimensions de couches), on y recopie les poids appris par le MLP PyTorch, puis on le convertit au format **TFLite** pour un déploiement embarqué (TinyML).

> Note : PyTorch n'exporte pas directement en TFLite. La voie la plus simple et lisible est de reconstruire une architecture Keras identique et d'y transférer les poids.

In [ ]:
import tensorflow as tf
from tensorflow import keras

# Reconstruction d'un MLP Keras équivalent au MLP PyTorch
# (mêmes tailles de couches : 128 -> 64 -> 32 -> 1, avec BatchNorm et Dropout)
keras_mlp = keras.Sequential([
    keras.layers.Input(shape=(n_features,)),
    keras.layers.Dense(128), keras.layers.BatchNormalization(), keras.layers.ReLU(), keras.layers.Dropout(0.2),
    keras.layers.Dense(64), keras.layers.BatchNormalization(), keras.layers.ReLU(), keras.layers.Dropout(0.2),
    keras.layers.Dense(32), keras.layers.BatchNormalization(), keras.layers.ReLU(), keras.layers.Dropout(0.2),
    keras.layers.Dense(1),
])
keras_mlp.summary()

In [ ]:
# Transfert des poids PyTorch -> Keras
# PyTorch stocke Linear.weight en (out, in) ; Keras Dense attend (in, out) -> transposition.
torch_layers = [m for m in mlp.net if isinstance(m, (nn.Linear, nn.BatchNorm1d))]
keras_layers = [l for l in keras_mlp.layers if isinstance(l, (keras.layers.Dense, keras.layers.BatchNormalization))]

for tl, kl in zip(torch_layers, keras_layers):
    if isinstance(tl, nn.Linear):
        W = tl.weight.detach().cpu().numpy().T   # (in, out)
        b = tl.bias.detach().cpu().numpy()
        kl.set_weights([W, b])
    else:  # BatchNorm1d -> BatchNormalization : gamma, beta, moving_mean, moving_var
        kl.set_weights([
            tl.weight.detach().cpu().numpy(),
            tl.bias.detach().cpu().numpy(),
            tl.running_mean.detach().cpu().numpy(),
            tl.running_var.detach().cpu().numpy(),
        ])
print("Poids transférés de PyTorch vers Keras.")

In [ ]:
# Conversion du modèle Keras en TFLite et sauvegarde sur disque
converter = tf.lite.TFLiteConverter.from_keras_model(keras_mlp)
converter.optimizations = [tf.lite.Optimize.DEFAULT]  # quantification dynamique (réduction de taille)
tflite_model = converter.convert()

with open("mlp_co2.tflite", "wb") as f:
    f.write(tflite_model)

print(f"Modèle TFLite exporté : mlp_co2.tflite ({len(tflite_model) / 1024:.1f} Ko)")

## 10. Prédictions vs valeurs réelles

Pour chaque modèle, on trace les prédictions en fonction des valeurs réelles. Plus les points sont proches de la diagonale (y = x), meilleur est le modèle.

In [ ]:
# Trois nuages de points (prédit vs réel) côte à côte
preds = {"MLP": pred_mlp, "Random Forest": pred_rf, "LSTM": pred_lstm}
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

lims = [y_test.min(), y_test.max()]
for ax, (name, pred) in zip(axes, preds.items()):
    ax.scatter(y_test, pred, alpha=0.3, s=10)
    ax.plot(lims, lims, "r--", label="y = x")  # diagonale idéale
    ax.set_title(f"{name}")
    ax.set_xlabel("CO2 réel (g/km)"); ax.set_ylabel("CO2 prédit (g/km)")
    ax.legend()

plt.tight_layout()
plt.show()